In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
import numpy as np
import tensorflow as tf

texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever",
    "Wouldn't recommend wasting your time",
    "Highly recommended for wasting time",
    "It was okay",
    "Nothing special about this film",
    "Average movie nothing more",
    "It was alright I guess"
]

In [3]:
# 0 = Negative, 1 = Positive, 2 = Neutral
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2])
labels = tf.keras.utils.to_categorical(labels, num_classes=3)  # one-hot encode

vocab_size = 1000
max_len = 6

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_len, padding="post")

print("Word Index:- ")
print(tokenizer.word_index)

print("\n Input Sequences:- \n", X)

Word Index:- 
{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'wasting': 9, 'time': 10, 'it': 11, 'was': 12, 'nothing': 13, 'love': 14, 'is': 15, 'amazing': 16, 'good': 17, 'excellent': 18, 'hate': 19, 'terrible': 20, 'boring': 21, 'worst': 22, 'ever': 23, "wouldn't": 24, 'recommend': 25, 'your': 26, 'highly': 27, 'recommended': 28, 'for': 29, 'okay': 30, 'special': 31, 'about': 32, 'average': 33, 'more': 34, 'alright': 35, 'guess': 36}

 Input Sequences:- 
 [[ 3 14  2  4  0  0]
 [ 2  5 15 16  0  0]
 [ 6 17  7  0  0  0]
 [18  8  0  0  0  0]
 [ 3 19  2  4  0  0]
 [20  5  0  0  0  0]
 [ 6 21  8  0  0  0]
 [22  7 23  0  0  0]
 [24 25  9 26 10  0]
 [27 28 29  9 10  0]
 [11 12 30  0  0  0]
 [13 31 32  2  5  0]
 [33  4 13 34  0  0]
 [11 12 35  3 36  0]]


In [4]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [5]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        attention_output = self.attention(inputs, inputs)
        out1 = self.layernorm1(inputs + attention_output)
        ffn_output = self.ffn(out1)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

In [19]:
embed_dim = 8
num_heads = 1
ff_dim = 32
inputs = layers.Input(shape=(max_len,))

x = TokenAndPositionEmbedding(
    max_length=max_len,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)

x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)

x = layers.GlobalAveragePooling1D()(x)

outputs = layers.Dense(3, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [20]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_4  │ (None, 6, 8)           │         8,048 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, 6, 8)           │           872 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,947 (34.95 KB)

 Trainable params: 8,947 (34.95 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.fit(X,labels,epochs=60,batch_size=2,verbose=1)

Epoch 1/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.3571 - loss: 1.1839    
Epoch 2/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5714 - loss: 0.9639     
Epoch 3/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7143 - loss: 0.8814 
Epoch 4/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7143 - loss: 0.8022 
Epoch 5/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8571 - loss: 0.7242 
Epoch 6/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7857 - loss: 0.6622 
Epoch 7/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8571 - loss: 0.5851 
Epoch 8/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8571 - loss: 0.5260 
Epoch 9/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9286 - loss: 0.4972 
Epoch 10/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9286 - loss: 0.4284 
Epoch 11/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.4065 
Epoch 12/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.

In [22]:
test_sentences = [
    "I love the film",
    "This movie was awful",
    "Rubbish movie",
    "Boring but Meaningful"
]

test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_len, padding="post")
predictions = model.predict(test_pad)

class_labels = ["Negative", "Positive", "Neutral"]

for sentence, prediction in zip(test_sentences, predictions):
    predicted_class = np.argmax(prediction)
    print(sentence, "->", prediction)
    print("Prediction:", class_labels[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
I love the film -> [0.01478097 0.94379824 0.04142082]
Prediction: Positive
This movie was awful -> [0.05336948 0.24350101 0.70312953]
Prediction: Neutral
Rubbish movie -> [0.292091   0.69246966 0.01543934]
Prediction: Positive
Boring but Meaningful -> [0.98772067 0.00844276 0.00383667]
Prediction: Negative
